# 5. Klasteryzacja i profilowanie segmentów

Klasteryzacja nie używa zmiennych docelowych (`SalaryUSD`, `CareerPlansThisYear`).
Liczbę klastrów K-Means wybieramy dla `k=2...10` przez inertia i silhouette.
TruncatedSVD jest oceniane przez explained variance; 2 komponenty służą wyłącznie do wizualizacji,
natomiast algorytmy korzystają z wielowymiarowej reprezentacji.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from salary_survey.config import DATA_PROCESSED_PATH, FIGURES_DIR, RANDOM_STATE
from salary_survey.pipelines import build_preprocessor

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
df = pd.read_csv(DATA_PROCESSED_PATH)
exclude = ["SalaryUSD", "SalaryUSD_Log", "CareerPlansThisYear", "Timestamp", "PostalCode"]
X_raw = df.drop(columns=exclude, errors="ignore")
preprocessor = build_preprocessor(X_raw)
X_encoded = preprocessor.fit_transform(X_raw)
print("Encoded clustering matrix:", X_encoded.shape)

## Explained variance i wybór liczby komponentów

In [ ]:
max_components = min(50, X_encoded.shape[1] - 1)
svd_full = TruncatedSVD(n_components=max_components, random_state=RANDOM_STATE)
X_svd_full = svd_full.fit_transform(X_encoded)
cumulative_variance = np.cumsum(svd_full.explained_variance_ratio_)
variance_table = pd.DataFrame({
    "component": np.arange(1, max_components + 1),
    "explained_variance": svd_full.explained_variance_ratio_,
    "cumulative_variance": cumulative_variance,
})
display(variance_table.head(15).round(4))

plt.figure(figsize=(10, 5))
plt.plot(variance_table["component"], variance_table["cumulative_variance"], marker="o", markersize=3)
plt.axhline(0.80, color="red", linestyle="--", label="80% wariancji")
plt.title("TruncatedSVD — skumulowana explained variance")
plt.xlabel("Liczba komponentów")
plt.ylabel("Skumulowany udział wyjaśnionej wariancji")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cluster_01_svd_explained_variance.png", dpi=180, bbox_inches="tight")
plt.show()
display(Markdown(
    "**Interpretacja.** Krzywa pokazuje koszt redukcji wymiaru. Dwa komponenty nie zachowują całej struktury, "
    "dlatego do klasteryzacji wykorzystujemy więcej wymiarów; komponenty 1–2 pozostają tylko mapą poglądową."
))

n_cluster_components = min(20, max_components)
X_cluster = StandardScaler().fit_transform(X_svd_full[:, :n_cluster_components])
print(f"Clustering uses {n_cluster_components} SVD components; 2D is visualization only.")

## K-Means: elbow curve i silhouette dla k=2...10

In [ ]:
k_rows = []
k_models = {}
for k in range(2, 11):
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_cluster)
    score = silhouette_score(
        X_cluster, labels, sample_size=min(4000, len(X_cluster)), random_state=RANDOM_STATE
    )
    min_cluster_pct = 100 * np.bincount(labels).min() / len(labels)
    k_rows.append({"k": k, "inertia": model.inertia_, "silhouette": score,
                   "min_cluster_pct": min_cluster_pct})
    k_models[k] = (model, labels)
k_scores = pd.DataFrame(k_rows)
display(k_scores.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.lineplot(data=k_scores, x="k", y="inertia", marker="o", ax=axes[0])
sns.lineplot(data=k_scores, x="k", y="silhouette", marker="o", ax=axes[1])
axes[0].set_title("Elbow curve")
axes[1].set_title("Silhouette score")
axes[0].set_xticks(range(2, 11))
axes[1].set_xticks(range(2, 11))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cluster_02_k_selection.png", dpi=180, bbox_inches="tight")
plt.show()

qualified = k_scores.loc[k_scores["min_cluster_pct"] >= 2.0]
best_k = int(qualified.loc[qualified["silhouette"].idxmax(), "k"])
kmeans, kmeans_labels = k_models[best_k]
df["Cluster_KMeans"] = kmeans_labels
display(Markdown(
    f"**Interpretacja.** Spośród rozwiązań bez klastrów mniejszych niż 2% próby najwyższy silhouette uzyskano dla k={best_k}. "
    "Wszystkie wartości silhouette są niskie, więc struktura segmentów jest słaba; ograniczenie minimalnego rozmiaru "
    "zapobiega wyborowi pozornie lepszego rozwiązania z kilkoma klastrami złożonymi z pojedynczych obserwacji."
))

## Wizualizacja 2D wybranego K-Means

In [ ]:
plot_frame = pd.DataFrame({
    "SVD_1": X_svd_full[:, 0], "SVD_2": X_svd_full[:, 1], "cluster": kmeans_labels.astype(str)
})
plt.figure(figsize=(10, 7))
sns.scatterplot(data=plot_frame.sample(min(6000, len(plot_frame)), random_state=RANDOM_STATE),
                x="SVD_1", y="SVD_2", hue="cluster", alpha=0.45, s=20, palette="tab10")
plt.title(f"K-Means (k={best_k}) — rzut na pierwsze 2 komponenty SVD")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cluster_03_kmeans_2d.png", dpi=180, bbox_inches="tight")
plt.show()
display(Markdown(
    "**Interpretacja.** Rzut 2D pomaga zobaczyć nakładanie segmentów, ale model był dopasowany w przestrzeni "
    f"{n_cluster_components}-wymiarowej. Pozorne nakładanie na wykresie nie oznacza automatycznie braku separacji w pozostałych wymiarach."
))

## Profile klastrów

In [ ]:
def mode_or_unknown(series):
    mode = series.dropna().mode()
    return mode.iloc[0] if len(mode) else "Unknown"

profiles = (df.groupby("Cluster_KMeans")
    .agg(
        size=("SalaryUSD", "size"),
        median_salary=("SalaryUSD", "median"),
        median_job_years=("YearsWithThisTypeOfJob", "median"),
        median_database_years=("YearsWithThisDatabase", "median"),
        dominant_country=("Country", mode_or_unknown),
        dominant_job=("JobTitle", mode_or_unknown),
        dominant_employment=("EmploymentStatus", mode_or_unknown),
    )
)
profiles["share_pct"] = 100 * profiles["size"] / profiles["size"].sum()
display(profiles.round(2))
profiles.to_csv(PROJECT_ROOT / "reports" / "cluster_profiles.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=profiles.reset_index(), x="Cluster_KMeans", y="median_salary", ax=axes[0])
sns.barplot(data=profiles.reset_index(), x="Cluster_KMeans", y="median_job_years", ax=axes[1])
axes[0].set_title("Mediana SalaryUSD w klastrach")
axes[1].set_title("Mediana stażu zawodowego w klastrach")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cluster_04_profiles.png", dpi=180, bbox_inches="tight")
plt.show()
display(Markdown(
    "**Interpretacja.** Profilowanie przez kraj, stanowisko, wynagrodzenie i staż nadaje segmentom treść. "
    "SalaryUSD nie uczestniczyło w klasteryzacji, więc różnice wynagrodzeń są opisem ex post, a nie wymuszoną separacją."
))

## DBSCAN: sprawdzenie parametrów i interpretacja wyniku negatywnego

In [ ]:
dbscan_rows = []
dbscan_models = {}
for eps in [0.5, 0.8, 1.1, 1.5, 2.0]:
    for min_samples in [10, 20, 40]:
        labels = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit_predict(X_cluster)
        non_noise = labels != -1
        n_clusters = len(set(labels[non_noise]))
        noise_pct = 100 * (~non_noise).mean()
        score = np.nan
        if n_clusters > 1 and non_noise.sum() > n_clusters:
            score = silhouette_score(
                X_cluster[non_noise], labels[non_noise],
                sample_size=min(3000, non_noise.sum()), random_state=RANDOM_STATE,
            )
        dbscan_rows.append({"eps": eps, "min_samples": min_samples,
                            "clusters": n_clusters, "noise_pct": noise_pct, "silhouette": score})
        dbscan_models[(eps, min_samples)] = labels
dbscan_results = pd.DataFrame(dbscan_rows)
display(dbscan_results.round(4))

valid = dbscan_results.dropna(subset=["silhouette"])
if valid.empty:
    dbscan_conclusion = (
        "Żaden sprawdzony wariant DBSCAN nie utworzył co najmniej dwóch stabilnych klastrów. "
        "To wynik negatywny: dane nie wykazują wyraźnej struktury gęstościowej przy badanym skalowaniu."
    )
else:
    best = valid.loc[valid["silhouette"].idxmax()]
    dbscan_conclusion = (
        f"Najlepszy badany DBSCAN: eps={best.eps}, min_samples={int(best.min_samples)}, "
        f"klastry={int(best.clusters)}, silhouette={best.silhouette:.3f}, szum={best.noise_pct:.1f}%."
    )
display(Markdown(f"**Interpretacja.** {dbscan_conclusion} DBSCAN nie jest przedstawiany jako udana segmentacja bez dowodów metrycznych."))

## Ograniczenia klasteryzacji

**Wniosek praktyczny.** K-Means daje słabą, ale opisywalną segmentację; DBSCAN nie potwierdził
obecności stabilnych klastrów. Wyniki mają charakter badawczy, a nie operacyjny.

- wyniki zależą od kodowania, skalowania oraz liczby komponentów SVD;
- silhouette mierzy separację geometryczną, nie użyteczność biznesową;
- K-Means preferuje klastry zbliżone do kulistych;
- profile są opisowe, a nie przyczynowe;
- negatywny wynik DBSCAN jest ważnym rezultatem i nie stanowi drugiej udanej segmentacji.